# Fine-Tuning Gemma 3 270M for Function Calling

This notebook demonstrates how to fine-tune the `google/gemma-3-270m-it` model for function calling using LoRA.

In [1]:
#!pip install -q -U transformers accelerate datasets peft trl

In [2]:
import os
from enum import Enum
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

set_seed(42)

/home/lmassaron/code/fine-tuning-workshop/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


In [3]:
class ChatmlSpecialTokens(str, Enum):
    """Enum class defining special tokens used in the ChatML format"""

    tools = "<tools>"
    eotools = "</tools>"
    think = "<think>"
    eothink = "</think>"
    tool_call = "<tool_call>"
    eotool_call = "</tool_call>"
    tool_response = "<tool_response>"
    eotool_response = "</tool_response>"
    pad_token = "<pad>"
    eos_token = "<eos>"

    @classmethod
    def list(cls):
        return [c.value for c in cls]

In [ ]:
class Config:
    model_name = "google/gemma-3-270m-it"
    dataset_name = "lmassaron/hermes-function-calling-v1"
    output_dir = "gemma-3-270M-it-function_calling"
    
    lora_arguments = {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "embed_tokens", "q_proj", "k_proj", "v_proj",
            "gate_proj", "up_proj", "down_proj", "o_proj", "lm_head"
        ],
        "bias": "none",
    }
    
    training_arguments = {
        "num_train_epochs": 1,
        "max_steps": -1, # Full 1 epoch run
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "max_length": 2048,
        "packing": True,
        "optim": "adamw_torch_fused",
        "learning_rate": 1e-4,
        "loss_type": "nll", 
        # we are telling SFTTrainer to bypass the new chunked memory optimization and fall back to the standard PyTorch Cross-Entropy Loss (nll).
        # This standard loss calculation is fully compatible with our LoRA-wrapped lm_head, allowing the training to run flawlessly
        "weight_decay": 0.1,
        "max_grad_norm": 1.0,
        "lr_scheduler_type": "cosine",
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "eval_strategy": "steps",
        "eval_steps": 100,
        "save_strategy": "steps",
        "save_steps": 100,
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "logging_steps": 100,
        "report_to": "none",
    }

config = Config()

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
    compute_dtype = torch.bfloat16
elif device.type == "mps":
    compute_dtype = torch.bfloat16 
else:
    compute_dtype = torch.float32

print(f"Using device: {device}, dtype: {compute_dtype}")

Using device: cuda, dtype: torch.bfloat16


In [6]:
# Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    pad_token=ChatmlSpecialTokens.pad_token.value,
    additional_special_tokens=ChatmlSpecialTokens.list(),
)

tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}{% if message['role'] != 'system' %}{{ '<start_of_turn>' + message['role'] + '\n' + message['content'] | trim + '<end_of_turn><eos>\n' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{'<start_of_turn>model\n'}}{% endif %}"

In [7]:
# Load Model
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=compute_dtype,
    #attn_implementation="eager",
    low_cpu_mem_usage=True,
    device_map=device, 
)

model.resize_token_embeddings(len(tokenizer))
print("Model loaded and token embeddings resized.")

Loading base model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded and token embeddings resized.


## Pre-Training Evaluation

Let's see how the baseline model responds to a prompt requiring a tool call.

In [8]:
eval_prompt = [
    {"role": "user", "content": '''You are a helpful assistant with access to the following tools:
<tools>
[{"name": "get_current_weather", "description": "Get the current weather in a given location", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}}]
</tools>
What's the weather like in Paris?'''}
]

text = tokenizer.apply_chat_template(eval_prompt, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)

print("--- BASE MODEL GENERATION ---")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False))

--- BASE MODEL GENERATION ---
Okay, I can provide you with the weather conditions in Paris.
<end_of_turn>


## Dataset & Training

We will now prepare the dataset and launch the training process using SFTTrainer.

In [9]:
print("Preparing dataset...")
def preprocess_and_filter(sample):
    messages = sample["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tokens = tokenizer.encode(text, truncation=False)
    
    if len(tokens) <= config.training_arguments["max_length"]:
        return {"text": text}
    else:
        return None

data = (
    load_dataset(config.dataset_name, split="train")
    .rename_column("conversations", "messages")
    .map(preprocess_and_filter, remove_columns="messages")
    .filter(lambda x: x is not None, keep_in_memory=False)
)

Preparing dataset...


Map:   0%|          | 0/4167 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4165 [00:00<?, ? examples/s]

In [10]:
dataset_train = data.train_test_split(test_size=0.2, shuffle=True, seed=0)
train_data = dataset_train["train"]
eval_data = dataset_train["test"]
print(f"Train size: {len(train_data)}, Validation size: {len(eval_data)}")

Train size: 3332, Validation size: 833


In [ ]:
model.config.use_cache = False

peft_config = LoraConfig(
    **config.lora_arguments,
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,
)

training_args = SFTConfig(
    output_dir=config.output_dir,
    dataset_text_field="text",
    **config.training_arguments,
    bf16 = compute_dtype == torch.bfloat16,
    fp16 = compute_dtype == torch.float16,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("Starting training process (max_steps=100)...")
trainer.train()

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported Flash Attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported Flash Attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Adding EOS to train dataset:   0%|          | 0/3332 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3332 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3332 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/3332 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/833 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/833 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Starting training process (max_steps=100)...


Step,Training Loss,Validation Loss


## Post-Training Evaluation

Now, let's see how the fine-tuned model responds to the same prompt.

In [ ]:
print("--- FINE-TUNED MODEL GENERATION ---")
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False))